# LSTM Training

Notebook ini melatih LSTM decoder dengan **6 variasi**:
- 3 variasi jumlah layer: 1, 2, 3
- 2 variasi ukuran hidden state: 128, 512

**Arsitektur:** Pre-inject (Show and Tell, Vinyals et al. 2015)  
**Loss:** Sparse Categorical Crossentropy  
**Optimizer:** Adam

In [ ]:
import os, sys, json, time, pickle
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

from src.lstm.train_keras import build_lstm_decoder, train, build_dataset, EMBED_DIM
from src.shared.caption_utils import load_vocabulary, clean_caption, encode_caption

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Load Feature & Vocabulary

In [ ]:
FEATURES_PATH = '../../features/flickr8k_features.npy'
FEAT_IDX_PATH = '../../features/flickr8k_idx.json'
VOCAB_PATH    = '../../features/vocab.json'
CAPTIONS_TXT  = '../../data/flickr8k/captions.txt'

features = np.load(FEATURES_PATH)
with open(FEAT_IDX_PATH) as f:
    img_idx = json.load(f)

image_ids = list(img_idx.keys())
FEATURE_DIM = features.shape[1]
print(f'Features shape: {features.shape}')

vocab = load_vocabulary(VOCAB_PATH)
VOCAB_SIZE = len(vocab)
print(f'Vocab size: {VOCAB_SIZE}')

## 2. Prepare Dataset

In [ ]:
MAX_LEN = 30

captions_dict = {}
with open(CAPTIONS_TXT) as f:
    for line in f:
        line = line.strip()
        if not line or line.lower().startswith('image'):
            continue
        parts = line.split(',', 1)
        if len(parts) < 2:
            continue
        img_file, caption = parts
        img_id = img_file.split('#')[0].strip()
        clean  = clean_caption(caption)
        captions_dict.setdefault(img_id, []).append(clean)

print(f'Total images with captions: {len(captions_dict)}')

captions_encoded = {}
for img_id, caps in captions_dict.items():
    captions_encoded[img_id] = [
        encode_caption(c, vocab, MAX_LEN) for c in caps
    ]

train_ids = image_ids[:6000]
val_ids   = image_ids[6000:7000]

train_captions = {k: captions_encoded[k] for k in train_ids if k in captions_encoded}
val_captions   = {k: captions_encoded[k] for k in val_ids   if k in captions_encoded}

print(f'Train images: {len(train_captions)}, Val images: {len(val_captions)}')

In [ ]:
train_ds = build_dataset(features, image_ids, train_captions,
                          batch_size=64, shuffle=True)
val_ds   = build_dataset(features, image_ids, val_captions,
                          batch_size=64, shuffle=False)

print('Datasets siap.')

for (feat_b, cap_b), tgt_b in train_ds.take(1):
    print('Feature batch shape:', feat_b.shape)
    print('Caption input batch shape:', cap_b.shape)
    print('Target batch shape:', tgt_b.shape)

## 3. Training 6 Variasi

In [ ]:
CONFIGS = [
    {'name': 'lstm-1layer-128', 'num_lstm_layers': 1, 'lstm_units': 128},
    {'name': 'lstm-2layer-128', 'num_lstm_layers': 2, 'lstm_units': 128},
    {'name': 'lstm-3layer-128', 'num_lstm_layers': 3, 'lstm_units': 128},
    {'name': 'lstm-1layer-512', 'num_lstm_layers': 1, 'lstm_units': 512},
    {'name': 'lstm-2layer-512', 'num_lstm_layers': 2, 'lstm_units': 512},
    {'name': 'lstm-3layer-512', 'num_lstm_layers': 3, 'lstm_units': 512},
]

histories = {}
os.makedirs('../../models/lstm', exist_ok=True)

for cfg in CONFIGS:
    sep = '=' * 60
    print(f'\n{sep}\nTraining: {cfg["name"]}\n{sep}')
    t0 = time.time()

    model = build_lstm_decoder(
        vocab_size=VOCAB_SIZE,
        feature_dim=FEATURE_DIM,
        embed_dim=EMBED_DIM,
        lstm_units=cfg['lstm_units'],
        num_lstm_layers=cfg['num_lstm_layers'],
        max_len=MAX_LEN - 1,
    )
    model.summary()

    hist = train(
        model, train_ds, val_ds,
        model_name=cfg['name'],
        epochs=20,
        save_path='../../models/lstm/',
    )
    histories[cfg['name']] = hist.history

    elapsed = time.time() - t0
    print(f'Selesai dalam {elapsed/60:.1f} menit')

with open('../../models/lstm/histories.pkl', 'wb') as f:
    pickle.dump(histories, f)
print('\nSemua variasi selesai. Histories tersimpan ke models/lstm/histories.pkl')

## 4. Plot Training & Validation Loss

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, cfg in zip(axes, CONFIGS):
    hist = histories[cfg['name']]
    ax.plot(hist['loss'],     label='Train Loss')
    ax.plot(hist['val_loss'], label='Val Loss', linestyle='--')
    ax.set_title(cfg['name'])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('LSTM Training & Validation Loss — Semua Variasi', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../../models/lstm/training_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print('Plot tersimpan ke models/lstm/training_curves.png')

## 5. Ringkasan Model

Tabel parameter count tiap variasi.

In [ ]:
print(f'{'Variasi':<22} {'Layers':<8} {'Units':<8} {'Best Val Loss':<16} {'Epochs'}')
print('-' * 65)
for cfg in CONFIGS:
    hist = histories[cfg['name']]
    best_val = min(hist['val_loss'])
    n_epochs  = len(hist['loss'])
    print(f'{cfg["name"]:<22} {cfg["num_lstm_layers"]:<8} {cfg["lstm_units"]:<8} {best_val:<16.4f} {n_epochs}')